In [46]:
import torch
import torch.nn as nn

# A dummy 3x3 image representing a '7'
with torch.no_grad():
    # BYT UT image_tensor MOT BILDEN DU LADDADE IN FRÅN DATORN:
    predictions = model(bild_tensor) 

predicted_number = torch.argmax(predictions).item()
print(f"Modellens råa poäng: {predictions.numpy()}")
print(f"Modellen gissade på index: {predicted_number}")

correct_answer = 1  # 1 means "Is a Seven"

print("Our 2D Tensor:")
image_tensor

Modellens råa poäng: [[-10.126283  10.376861]]
Modellen gissade på index: 1
Our 2D Tensor:


tensor([[255., 255., 255.],
        [  0.,   0., 255.],
        [  0.,   0., 255.]])

In [47]:
class SimpleNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        # 784 ingångar (28x28 pixlar), 2 utgångar (Klass 0 eller Klass 1)
        self.layer = nn.Linear(784, 2) 
        
    def forward(self, x):
        x = x.unsqueeze(0) # Lägg till batch-dimensionen [1, 1, 28, 28]
        x = self.flatten(x)
        return self.layer(x)

# Starta en ny tom modell
model = SimpleNetwork()
print("Modellen är uppdaterad för 28x28-bilder!")

Modellen är uppdaterad för 28x28-bilder!


In [48]:
with torch.no_grad():
    predictions = model(bild_tensor)

predicted_number = torch.argmax(predictions).item()

print(f"Modellens råa poäng: {predictions.numpy()}")
print(f"Modellen gissade på index: {predicted_number}")
print("-" * 30)

if predicted_number == correct_answer:
    print("🎯 RÄTT! Modellen gissade rätt av ren slump.")
else:
    print("❌ FEL! Vikterna är slumpmässiga, så den gissade fel. Detta är helt normalt!")


Modellens råa poäng: [[-1.2324771  -0.02527354]]
Modellen gissade på index: 1
------------------------------
🎯 RÄTT! Modellen gissade rätt av ren slump.


In [49]:
# 1. Definiera verktyg för inlärning (Loss-funktion och Optimizer)
criterion = nn.CrossEntropyLoss()
# SGD är algoritmen som skruvar på nätverkets interna vikter
optimizer = torch.optim.SGD(model.parameters(), lr=0.1) 

print("Tränar modellen på vår 2D-tensor...")

# Vi låter modellen titta på bilden och rätta sig själv 20 gånger (epochs)
for epoch in range(20):
    optimizer.zero_grad() # Nollställ gamla korrigeringar
    
    outputs = model(bild_tensor) # Gör en gissning
    
    # Räkna ut HUR FEL modellen hade (loss) genom att jämföra med rätt svar
    target = torch.tensor([correct_answer])
    loss = criterion(outputs, target)
    
    loss.backward() # Räkna ut hur mycket varje matematisk vikt måste ändras
    optimizer.step() # Uppdatera vikterna!
    
    # Skriv ut felmarginalen var 5:e omgång
    if (epoch + 1) % 5 == 0:
        print(f"Omgång {epoch+1}/20 - Felmarginal (Loss): {loss.item():.4f}")

print("\nTräning klar! Modellen har justerat sin matematik.")


Tränar modellen på vår 2D-tensor...
Omgång 5/20 - Felmarginal (Loss): 0.0000
Omgång 10/20 - Felmarginal (Loss): 0.0000
Omgång 15/20 - Felmarginal (Loss): 0.0000
Omgång 20/20 - Felmarginal (Loss): 0.0000

Träning klar! Modellen har justerat sin matematik.


In [50]:
# 1. SKAPA EN NY BILD (Detta är INTE en 7:a)
# Ett mönster med ett streck i mitten
new_image = torch.tensor([
    [  0.0,   0.0,   0.0],
    [255.0, 255.0, 255.0],
    [  0.0,   0.0,   0.0]
])

print("Vår nya okända bild:")
print(new_image)
print("-" * 30)

# Säg till PyTorch att inte räkna ut några gradienter (vi bara testar/gissar)
with torch.no_grad():
    predictions = model(bild_tensor)

predicted_class = torch.argmax(predictions).item()

print(f"Råa poäng från modellen: {predictions.numpy()}")
print(f"🤖 AI gissar på klass: {predicted_class}")
print("-" * 40)
print("Notera: Modellen är helt otränad för denna bildstorlek,")
print("så den väljer bara en klass helt slumpmässigt baserat på sina startvikter!")


Vår nya okända bild:
tensor([[  0.,   0.,   0.],
        [255., 255., 255.],
        [  0.,   0.,   0.]])
------------------------------
Råa poäng från modellen: [[-18.166801  16.90905 ]]
🤖 AI gissar på klass: 1
----------------------------------------
Notera: Modellen är helt otränad för denna bildstorlek,
så den väljer bara en klass helt slumpmässigt baserat på sina startvikter!


In [51]:
import torch
from PIL import Image
import torchvision.transforms as transforms

# 1. HÄMTA EN BILD
# Byt ut 'min_bild.jpg' mot namnet på en bild du har i din projektmapp!
# (Eller använd en webbkamerabild om du sparar ner den som en fil först)
bild_bana = 'tensors.png' 

try:
    # Öppna bilden med Pillow
    original_bild = Image.open(bild_bana).convert('L')
    
    # 2. TRANSFORMERA BILDEN TILL DET FORMAT AI:N KRÄVER
    transformering = transforms.Compose([
        transforms.Resize((28, 28)),          # Skala ner bilden till t.ex. 28x28 pixlar
        transforms.ToTensor(),                # Gör om till Tensor (och skalar pixelvärden till 0.0 - 1.0)
    ])
    
    bild_tensor = transformering(original_bild)
    
    print("--- Bilden är nu en Tensor! ---")
    print("Tensorns form (Shape):", bild_tensor.shape)
    
except FileNotFoundError:
    print(f"❌ Hittade inte filen '{bild_bana}'.")
    print("Lägg en bild i din projektmapp till vänster i VS Code och döp om texten i koden till rätt filnamn!")


--- Bilden är nu en Tensor! ---
Tensorns form (Shape): torch.Size([1, 28, 28])


In [52]:
import torchvision
import torchvision.transforms as transforms

# 1. LADDA NER TUSENTALS RIKTIGA 28x28 BILDER PÅ SIFFROR (MNIST)
transformering = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])

# Hämtar träningsbilder (60 000 stycken!)
train_set = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transformering)
train_loader = torch.utils.data.DataLoader(train_set, batch_size=64, shuffle=True)

print(f"🔥 Datasetet är nedladdat! Vi har nu {len(train_set)} bilder att träna på.")


100.0%
100.0%
100.0%
100.0%

🔥 Datasetet är nedladdat! Vi har nu 60000 bilder att träna på.


In [53]:
import torch.nn as nn
import torch.optim as optim

# 1. DEFINIERA MODELLEN FÖR RIKTIG SIFFERIGENKÄNNING (0-9)
class MNISTNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        # 784 ingångar (28x28 pixlar), 10 utgångar (en poäng för varje siffra 0-9)
        self.layer1 = nn.Linear(784, 128) # Ett dolt lager som hittar mönster
        self.relu = nn.ReLU()             # Aktiveringsfunktion för komplexitet
        self.layer2 = nn.Linear(128, 10)  # Slutlager med 10 utgångar
        
    def forward(self, x):
        x = self.flatten(x)
        x = self.layer1(x)
        x = self.relu(x)
        return self.layer2(x)

# Starta den nya modellen och ställ in verktygen
mnist_model = MNISTNetwork()
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(mnist_model.parameters(), lr=0.01)

print("Tränar påbörjad på 60 000 bilder...")

# 2. TRÄNA MODELLEN UNDER EN "EPOCH" (Går igenom alla bilder en gång)
mnist_model.train()
for batch_idx, (images, labels) in enumerate(train_loader):
    optimizer.zero_grad()
    
    # Här skickas en 4D-tensor in! Shape: [64, 1, 28, 28]
    outputs = mnist_model(images)
    
    loss = criterion(outputs, labels)
    loss.backward()
    optimizer.step()
    
    # Skriv ut framsteg var 200:e grupp
    if batch_idx % 200 == 0:
        print(f"Bilder bearbetade: {batch_idx * len(images)}/60000 - Felmarginal (Loss): {loss.item():.4f}")

print("\nTräning klar! Modellen kan nu känna igen handskrivna siffror 0-9.")


Tränar påbörjad på 60 000 bilder...
Bilder bearbetade: 0/60000 - Felmarginal (Loss): 2.3057
Bilder bearbetade: 12800/60000 - Felmarginal (Loss): 0.8639
Bilder bearbetade: 25600/60000 - Felmarginal (Loss): 0.5641
Bilder bearbetade: 38400/60000 - Felmarginal (Loss): 0.6350
Bilder bearbetade: 51200/60000 - Felmarginal (Loss): 0.2698

Träning klar! Modellen kan nu känna igen handskrivna siffror 0-9.
